# RI-JK RHF Hessian：CP-HF 分解 (1) 结算计算、等式右计算

In [1]:
from pyscf import gto, scf, lib, df, hessian
import numpy as np
from functools import partial
from pyscf.df.grad.rhf import _int3c_wrapper

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = scf.RHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_r_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_r_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_r_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)

de_ref shape: (4, 4, 3, 3)


In [5]:
de_cphf = np.load("nh3_r_hf_decomp.npz")["de_cphf"]

In [6]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao, nmo = mo_coeff.shape
mocc = mo_coeff[:, mo_occ > 0]
occ_occupation = mo_occ[mo_occ > 0]
nocc = mocc.shape[1]
dm0 = np.dot(mocc, mocc.T) * 2
dme0 = np.einsum('pi,qi,i->pq', mocc, mocc, mo_energy[mo_occ > 0]) * 2
natm = mol.natm
atmlst = range(natm)
aoslices = mol.aoslice_by_atom()
aux = mf.with_df.auxmol
auxslices = aux.aoslice_by_atom()
naux = aux.nao
mocc_2 = np.einsum("pi,i->pi", mocc, occ_occupation**0.5)
occ_energy = mo_energy[mo_occ > 0]

In [7]:
eocc = mo_energy[mo_occ > 0]
evir = mo_energy[mo_occ == 0]
mvir = mo_coeff[:, mo_occ == 0]

## Overview

In [8]:
def ovlp_deriv1_generator(mol):
    int1e_ipovlp = mol.intor("int1e_ipovlp")
    
    def get_ovlp_deriv_at_atoms(A):
        shl0, shl1, p0, p1 = aoslices[A]
        s1ao = np.zeros((3, nao, nao))
        s1ao[:, p0:p1, :] += - int1e_ipovlp[:, p0:p1] 
        s1ao[:, :, p0:p1] += - int1e_ipovlp[:, p0:p1].transpose(0, 2, 1)
        return s1ao
    return get_ovlp_deriv_at_atoms

In [9]:
f1ao = mf_hess.make_h1(mo_coeff, mo_occ)
mo1, mo_e1 = mf_hess.solve_mo1(mo_energy, mo_coeff, mo_occ, f1ao)
mo1 = np.array(mo1)
mo_e1 = np.array(mo_e1)
int1e_ipovlp = mol.intor("int1e_ipovlp")
de = np.zeros((natm, natm, 3, 3))

for i0, ia in enumerate(atmlst):
    shl0, shl1, p0, p1 = aoslices[ia]
    s1ao = ovlp_deriv1_generator(mol)(ia)
    # s1oo = np.einsum('tuv, ui, vj -> tij', s1ao, mocc, mocc)
    s1oo = mocc.T @ s1ao @ mocc

    for j0 in range(i0+1):
        ja = atmlst[j0]
        q0, q1 = aoslices[ja][2:]
        # dm1 = np.einsum('sui, vi -> suv', mo1[ja], mocc)
        dm1 = mo1[ja] @ mocc.T
        # de[i0,j0] += 4 * np.einsum('tuv, suv -> ts', h1ao[ia], dm1)
        de[i0,j0] += 4 * (f1ao[ia][:, None] * dm1[None, :]).sum(axis=(-1, -2))
        # dm1 = np.einsum('sui, vi, i -> suv', mo1[ja], mocc, occ_energy)
        dm1 = mo1[ja] @ (mocc * occ_energy).T
        # de[i0,j0] -= 4 * np.einsum('tuv, suv -> ts', s1ao, dm1)
        de[i0,j0] -= 4 * (s1ao[:, None] * dm1[None, :]).sum(axis=(-1, -2))
        # de[i0,j0] -= 2 * np.einsum('tuv, suv -> ts', s1oo, mo_e1[ja])
        de[i0,j0] -= 2 * (s1oo[:, None] * mo_e1[ja][None, :]).sum(axis=(-1, -2))
        
    for j0 in range(i0):
        de[j0,i0] = de[i0,j0].T

np.allclose(de, de_cphf)

True

In [10]:
int2c2e = aux.intor("int2c2e")
int2c2e_inv = np.linalg.inv(int2c2e)
int2c2e_ip1 = aux.intor("int2c2e_ip1")
int3c2e = _int3c_wrapper(mol, aux, "int3c2e", "s1")()
int3c2e_ip1 = _int3c_wrapper(mol, aux, "int3c2e_ip1", "s1")().reshape([3, nao, nao, naux])
int3c2e_ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip2", "s1")().reshape([3, nao, nao, naux])

## f1ao

### Reference values

This part will give the reference values of what we need to construct.

In [11]:
mf_hess_aux0 = mf.Hessian()
mf_hess_aux0.auxbasis_response = 0
ref0 = list(df.hessian.rhf._gen_jk(mf_hess_aux0, mo_coeff, mo_occ))
h1ao = np.array([r[1] for r in ref0])
j1ao_aux0 = np.array([r[2] for r in ref0])
k1ao_aux0 = np.array([r[3] for r in ref0])

In [12]:
mf_hess_aux1 = mf.Hessian()
mf_hess_aux1.auxbasis_response = 1
ref1 = list(df.hessian.rhf._gen_jk(mf_hess_aux1, mo_coeff, mo_occ))
j1ao_aux1 = np.array([r[2] for r in ref1]) - j1ao_aux0
k1ao_aux1 = np.array([r[3] for r in ref1]) - k1ao_aux0

In [13]:
assert np.allclose(h1ao + j1ao_aux0 + j1ao_aux1 - 0.5 * (k1ao_aux0 + k1ao_aux1), f1ao)

### h1ao part

This part is actually exactly the same to the gradient part.

In [14]:
def hcore_deriv1_generator(mol):
    h1 = - mol.intor("int1e_ipkin") - mol.intor("int1e_ipnuc")
    if mol.has_ecp():
        h1 -= mol.intor("ECPscalar_ipnuc")
    ecp_atoms = set(mol._ecpbas[:, gto.ATOM_OF])
    aoslices = mol.aoslice_by_atom()
    
    def get_hcore_deriv_at_atoms(A):
        _, _, p0, p1 = aoslices[A]
        z = mol.atom_charge(A)
        with mol.with_rinv_at_nucleus(A):
            h1ao = -z * mol.intor("int1e_iprinv")
            if A in ecp_atoms:
                h1ao += mol.intor("ECPscalar_iprinv")
        h1ao[:, p0:p1] += h1[:, p0:p1]
        return h1ao + h1ao.swapaxes(-1, -2)
    return get_hcore_deriv_at_atoms

In [15]:
for A in range(mol.natm):
    assert np.allclose(hcore_deriv1_generator(mol)(A), h1ao[A])

### j1ao part

In [16]:
%%time
scr1 = np.einsum("tuvP, PQ, klQ, kl -> tuv", int3c2e_ip1, int2c2e_inv, int3c2e, dm0)

j1ao_aux0_recap = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = aoslices[A]
    slcA = slice(p0, p1)
    # (10|0)(0|00)
    j1ao_aux0_recap[A, :, slcA, :] -= scr1[:, slcA, :]
    # (01|0)(0|00) (can be symmetrized)
    j1ao_aux0_recap[A, :, :, slcA] -= scr1[:, slcA, :].swapaxes(-1, -2)
    # (00|0)(0|10), (00|0)(0|01)
    scr2 = np.einsum("tklP, PQ, uvQ, kl -> tuv", int3c2e_ip1[:, slcA], int2c2e_inv, int3c2e, dm0[slcA])
    j1ao_aux0_recap[A] -= 2 * scr2
assert np.allclose(j1ao_aux0_recap, j1ao_aux0)

CPU times: user 422 ms, sys: 10.1 ms, total: 432 ms
Wall time: 32.6 ms


In [17]:
%%time
j1ao_aux1_recap = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = auxslices[A]
    slcA = slice(p0, p1)
    # (00|1)(0|00)
    j1ao_aux1_recap[A] -= np.einsum("tuvP, PQ, klQ, kl -> tuv", int3c2e_ip2[:, :, :, slcA], int2c2e_inv[slcA, :], int3c2e, dm0)
    # (00|0)(1|00)
    j1ao_aux1_recap[A] -= np.einsum("uvP, PQ, tklQ, kl -> tuv", int3c2e, int2c2e_inv[:, slcA], int3c2e_ip2[:, :, :, slcA], dm0)
    # (00|0)(1|0)(0|00)
    j1ao_aux1_recap[A] += np.einsum("uvP, PQ, tQR, RS, klS, kl -> tuv", int3c2e, int2c2e_inv[:, slcA], int2c2e_ip1[:, slcA], int2c2e_inv, int3c2e, dm0)
    # (00|0)(0|1)(0|00)
    j1ao_aux1_recap[A] += np.einsum("uvP, PQ, tRQ, RS, klS, kl -> tuv", int3c2e, int2c2e_inv, int2c2e_ip1[:, slcA], int2c2e_inv[slcA, :], int3c2e, dm0)
assert np.allclose(j1ao_aux1_recap, j1ao_aux1)

CPU times: user 279 ms, sys: 12.8 ms, total: 292 ms
Wall time: 21.5 ms


### k1ao part

In [18]:
scr1 = np.einsum("tuvP, PQ, klQ, vi, li -> tuk", int3c2e_ip1, int2c2e_inv, int3c2e, mocc_2, mocc_2)

k1ao_aux0_recap = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = aoslices[A]
    slcA = slice(p0, p1)
    # (10|0)(0|00)
    k1ao_aux0_recap[A, :, slcA, :] -= scr1[:, slcA, :]
    # (01|0)(0|00)
    k1ao_aux0_recap[A, :, :, slcA] -= scr1[:, slcA, :].swapaxes(-1, -2)
    # (00|0)(0|10), (00|0)(0|01)
    scr2 = np.einsum("tklP, PQ, uvQ, ki, ui -> tlv", int3c2e_ip1[:, slcA], int2c2e_inv, int3c2e, mocc_2[slcA], mocc_2)
    k1ao_aux0_recap[A] -= scr2 + scr2.swapaxes(-1, -2)
assert np.allclose(k1ao_aux0_recap, k1ao_aux0)

In [19]:
# this part of computation is not sutiable using dm-only when einsum
k1ao_aux1_recap = np.zeros([natm, 3, nao, nao])
for A in range(mol.natm):
    sh0, sh1, p0, p1 = auxslices[A]
    slcA = slice(p0, p1)
    # (00|1)(0|00)
    k1ao_aux1_recap[A] -= np.einsum("tuvP, PQ, klQ, vi, li -> tuk", int3c2e_ip2[:, :, :, slcA], int2c2e_inv[slcA, :], int3c2e, mocc_2, mocc_2)
    # (00|0)(1|00)
    k1ao_aux1_recap[A] -= np.einsum("uvP, PQ, tklQ, vi, li -> tuk", int3c2e, int2c2e_inv[:, slcA], int3c2e_ip2[:, :, :, slcA], mocc_2, mocc_2)
    # (00|0)(1|0)(0|00)
    k1ao_aux1_recap[A] += np.einsum("uvP, PQ, tQR, RS, klS, vi, li -> tuk", int3c2e, int2c2e_inv[:, slcA], int2c2e_ip1[:, slcA], int2c2e_inv, int3c2e, mocc_2, mocc_2)
    # (00|0)(0|1)(0|00)
    k1ao_aux1_recap[A] += np.einsum("uvP, PQ, tRQ, RS, klS, vi, li -> tuk", int3c2e, int2c2e_inv, int2c2e_ip1[:, slcA], int2c2e_inv[slcA, :], int3c2e, mocc_2, mocc_2)
assert np.allclose(k1ao_aux1_recap, k1ao_aux1)

## Response Function

CP-HF/KS need response function (instead of usual fock/veff). However, for Hartree-Fock problem, response (F) is identical to potential (V).

This part need to be expanded for KS counterpart.

In [20]:
vresp = mf.gen_response()

In [21]:
dm_rand = np.random.rand(nao, nao)
dm_rand += dm_rand.T
np.allclose(vresp(dm_rand), mf.get_veff(dm=dm_rand))

True

In [22]:
dmo1 = mo_coeff @ mo1 @ mocc.T
dmo1 = 0.5 * (dmo1 + dmo1.swapaxes(-1, -2))

### CP-HF recover

In [23]:
def gen_vind(mf, mo_coeff, mo_occ):
    nao, nmo = mo_coeff.shape
    mocc = mo_coeff[:,mo_occ>0]
    nocc = mocc.shape[1]
    vresp = mf.gen_response(mo_coeff, mo_occ, hermi=1)
    def fx(mo1):
        mo1 = mo1.reshape(-1,nmo,nocc)
        nset = len(mo1)
        dm1 = np.empty((nset,nao,nao))
        for i, x in enumerate(mo1):
            dm = mo_coeff @ (x*2) @ mocc.T
            dm1[i] = dm + dm.T
        v1 = vresp(dm1)
        v1vo = np.empty_like(mo1)
        for i, x in enumerate(v1):
            v1vo[i] = mo_coeff.T @ x @ mocc
        return v1vo
    return fx


In [24]:
def solve_withs1(
    fvind, mo_energy, mo_occ, h1, s1, max_cycle=50, tol=1e-9, level_shift=0
):
    """For field dependent basis. First order overlap matrix is non-zero.
    The first order orbitals are set to
    C^1_{ij} = -1/2 S1
    e1 = h1 - s1*e0 + (e0_j-e0_i)*c1 + vhf[c1]

    Kwargs:
        level_shift : float
            Add to diagonal terms to slightly improve the convergence speed of
            Krylov solver

    Returns:
        First order orbital coefficients (in MO basis) and first order orbital
        energy matrix
    """
    occidx = mo_occ > 0
    viridx = mo_occ == 0
    e_a = mo_energy[viridx]
    e_i = mo_energy[occidx]
    e_ai = 1 / (e_a[:, None] + level_shift - e_i)
    nvir, nocc = e_ai.shape
    nmo = nocc + nvir

    s1 = s1.reshape(-1, nmo, nocc)
    hs = mo1base = h1.reshape(-1, nmo, nocc) - s1 * e_i

    mo1base = hs.copy()
    mo1base[:, viridx] *= -e_ai
    mo1base[:, occidx] = -s1[:, occidx] * 0.5

    def vind_vo(mo1):
        mo1 = mo1.reshape(-1, nmo, nocc)
        v = fvind(mo1).reshape(-1, nmo, nocc)
        if level_shift != 0:
            v -= mo1 * level_shift
        v[:, viridx, :] *= e_ai
        v[:, occidx, :] = 0
        return v.reshape(-1, nmo * nocc)

    mo1 = lib.krylov(
        vind_vo,
        mo1base.reshape(-1, nmo * nocc),
        tol=tol,
        max_cycle=max_cycle,
    )
    mo1 = mo1.reshape(-1, nmo, nocc)
    mo1[:, occidx] = mo1base[:, occidx]

    hs += fvind(mo1).reshape(-1, nmo, nocc)
    mo1[:, viridx] = hs[:, viridx] / (e_i - e_a[:, None])

    # mo_e1 has the same symmetry as the first order Fock matrix (hermitian or
    # anti-hermitian). mo_e1 = v1mo - s1*lib.direct_sum('i+j->ij',e_i,e_i)
    mo_e1 = hs[:, occidx, :]
    mo_e1 += mo1[:, occidx] * (e_i[:, None] - e_i)

    if h1.ndim == 3:
        return mo1, mo_e1
    else:
        assert h1.ndim == 2
        return mo1[0], mo_e1[0]

In [25]:
f1ao = np.array(f1ao)
s1ao = np.array([ovlp_deriv1_generator(mol)(A) for A in range(mol.natm)])
f1mo = mo_coeff.T @ f1ao @ mocc
s1mo = mo_coeff.T @ s1ao @ mocc

In [26]:
vind = gen_vind(mf, mo_coeff, mo_occ)
t1, t2 = solve_withs1(vind, mo_energy, mo_occ, f1mo.reshape(-1, nao, nocc), s1mo.reshape(-1, nao, nocc))

In [27]:
np.allclose(mo_coeff @ t1, mo1.reshape(-1, nao, nocc))

True